# 05 - Paper Tables

Generates the LaTeX for every results table in the paper from
`results/all_evaluations.json` and `data/stats/`.

Run `04_evaluation.ipynb` first. Output goes to `paper/tables/`.

In [ ]:
import json, collections
from pathlib import Path

ROOT    = Path.cwd().parent
RESULTS = ROOT / "results"
STATS   = ROOT / "data" / "stats"
OUT     = ROOT / "paper" / "tables"
OUT.mkdir(parents=True, exist_ok=True)

res = json.loads((RESULTS / "all_evaluations.json").read_text(encoding="utf-8"))
print("loaded:", ", ".join(res.keys()))

## Per-tag tables

In [ ]:
def pertag_table(block, caption, label):
    rows = []
    for t, v in sorted(block["per_tag"].items()):
        rows.append("%-6s & %.2f & %.2f & %.2f & %s \\\\"
                    % (t, v["precision"], v["recall"], v["f1"],
                       format(v["support"], ",")))
    n = format(block["tokens"], ",")
    macro_p = sum(v["precision"] for v in block["per_tag"].values()) / len(block["per_tag"])
    macro_r = sum(v["recall"] for v in block["per_tag"].values()) / len(block["per_tag"])
    wp = sum(v["precision"] * v["support"] for v in block["per_tag"].values()) / block["tokens"]
    wr = sum(v["recall"] * v["support"] for v in block["per_tag"].values()) / block["tokens"]
    return "\n".join([
        r"\begin{table}[t]", r"\centering",
        r"\caption{%s}" % caption, r"\label{%s}" % label,
        r"\begin{tabular}{lcccr}", r"\hline",
        r"\textbf{PoS Tag} & \textbf{Precision} & \textbf{Recall} & "
        r"\textbf{$F_1$} & \textbf{Support} \\", r"\hline",
        "\n".join(rows), r"\hline",
        "Macro avg.    & %.2f & %.2f & %.2f & %s \\\\"
        % (macro_p, macro_r, block["macro_f1"], n),
        "Weighted avg. & %.2f & %.2f & %.2f & %s \\\\"
        % (wp, wr, block["weighted_f1"], n),
        r"\hline", r"\end{tabular}", r"\end{table}",
    ])


t1 = pertag_table(res["lexicon_baseline"],
                  "Per-tag performance of the lexicon baseline on the held-out test set.",
                  "tab:lexicon")
t2 = pertag_table(res["bilstm_crf"],
                  "Per-tag performance of the BiLSTM--CRF tagger on the held-out test set.",
                  "tab:bilstm")

(OUT / "table_lexicon.tex").write_text(t1, encoding="utf-8")
(OUT / "table_bilstm.tex").write_text(t2, encoding="utf-8")
print(t2)

## Comparison table

In [ ]:
def macro_excl(block, drop):
    v = [m["f1"] for t, m in block["per_tag"].items() if t != drop]
    return sum(v) / len(v)

lex, nn = res["lexicon_baseline"], res["bilstm_crf"]
cmp_tex = "\n".join([
    r"\begin{table}[t]", r"\centering",
    r"\caption{Lexicon baseline against the BiLSTM--CRF on the held-out test set.}",
    r"\label{tab:compare}",
    r"\begin{tabular}{lcc}", r"\hline",
    r"\textbf{Metric} & \textbf{Lexicon} & \textbf{BiLSTM--CRF} \\", r"\hline",
    r"Token accuracy (\%%)   & %.2f & %.2f \\" % (100 * lex["accuracy"], 100 * nn["accuracy"]),
    r"Macro $F_1$           & %.3f & %.3f \\" % (lex["macro_f1"], nn["macro_f1"]),
    r"Macro $F_1$ excl.\ \texttt{NUM} & %.3f & %.3f \\"
    % (macro_excl(lex, "NUM"), macro_excl(nn, "NUM")),
    r"Weighted $F_1$        & %.3f & %.3f \\" % (lex["weighted_f1"], nn["weighted_f1"]),
    r"\hline", r"\end{tabular}", r"\end{table}",
])
(OUT / "table_comparison.tex").write_text(cmp_tex, encoding="utf-8")
print(cmp_tex)

## Tag distribution

In [ ]:
core = json.loads((STATS / "dataset_stats_final.json").read_text(
    encoding="utf-8"))["tag_distribution"]
CT = sum(core.values())

# The training split is not redistributed; its tag counts are released instead.
train_counts = collections.Counter(json.loads(
    (STATS / "train_tag_counts.json").read_text(encoding="utf-8"))["tag_counts"])
TT = sum(train_counts.values())
print("training tokens: %s   core tokens: %s" % (format(TT, ","), format(CT, ",")))

rows = []
for t, n in train_counts.most_common():
    cn = core.get(t, 0)
    rows.append("%-6s & %9s & %5.2f & %7s & %5.2f \\\\"
                % (t, format(n, ","), 100 * n / TT,
                   format(cn, ","), 100 * cn / CT))

dist = "\n".join([
    r"\begin{table}[t]", r"\centering",
    r"\caption{Tag distribution in the training corpus and in the manually reviewed core.}",
    r"\label{tab:tagdist}",
    r"\begin{tabular}{lrrrr}", r"\hline",
    r"& \multicolumn{2}{c}{\textbf{Training corpus}} & "
    r"\multicolumn{2}{c}{\textbf{Reviewed core}} \\",
    r"\cline{2-3}\cline{4-5}",
    r"\textbf{Tag} & \textbf{Tokens} & \textbf{\%} & "
    r"\textbf{Tokens} & \textbf{\%} \\", r"\hline",
    "\n".join(rows), r"\hline",
    "Total & %s & & %s & \\\\" % (format(TT, ","), format(CT, ",")),
    r"\hline", r"\end{tabular}", r"\end{table}",
])
(OUT / "table_tagdist.tex").write_text(dist, encoding="utf-8")
print(dist)

## External gold results

In [ ]:
ea, en = res["external_raw_nopunct_clean"], res["external_scheme_neutral"]
ext = "\n".join([
    r"\begin{table}[t]", r"\centering",
    r"\caption{Zero-shot performance on the gold data of \citet{ghosh2025mizo}.}",
    r"\label{tab:external}",
    r"\begin{tabular}{lcc}", r"\hline",
    r"\textbf{Metric} & \textbf{Original tagset} & \textbf{Scheme-neutral} \\",
    r"\hline",
    r"Tokens               & %s & %s \\" % (format(ea["tokens"], ","),
                                            format(en["tokens"], ",")),
    r"Token accuracy (\%%)  & %.2f  & %.2f \\" % (100 * ea["accuracy"],
                                                 100 * en["accuracy"]),
    r"Macro $F_1$          & %.3f  & %.3f \\" % (ea["macro_f1"], en["macro_f1"]),
    r"Weighted $F_1$       & %.3f  & %.3f \\" % (ea["weighted_f1"], en["weighted_f1"]),
    r"\hline", r"\end{tabular}", r"\end{table}",
])
(OUT / "table_external.tex").write_text(ext, encoding="utf-8")
print(ext)

print("\nAll tables written to", OUT)
for p in sorted(OUT.glob("*.tex")):
    print("  ", p.name)